In [2]:
import subprocess
import glob
import os
import re
import struct
import pandas as pd

KTEST_TOOL = "/mnt/data2/EMULATOR/misc/klee/build/bin/ktest-tool"
OUT_DIR = "out_spike_sfence_vma"

def parse_ktest(path):
    out = subprocess.check_output([KTEST_TOOL, path], text=True, errors="ignore")
    data = {}
    current_name = None

    for line in out.splitlines():
        line = line.strip()

        m = re.match(r"object \d+: name: '(.+)'", line)
        if m:
            current_name = m.group(1)
            continue

        m = re.match(r"object \d+: hex\s*:\s*0x([0-9a-fA-F]+)", line)
        if m and current_name:
            data[current_name] = bytes.fromhex(m.group(1))
            continue

    return data

def read_i32(raw):
    if raw is None:
        return None
    return struct.unpack("<i", raw[:4])[0]

def exc_name(x):
    if x == 0:
        return "none"
    if x == 2:
        return "illegal instruction"
    if x == 22:
        return "virtual instruction"
    return f"unknown {x}"

def explain(row):
    if row["ext_s"] == 0:
        return "S extension disabled -> illegal instruction, no TLB flush"

    if row["has_mmu"] == 0:
        return "MMU disabled -> illegal instruction, no TLB flush"

    if row["v"] == 1:
        if row["prv"] == 0:
            return "virtual mode and U privilege -> virtual instruction, no TLB flush"
        if row["hstatus_vtvm"] == 1:
            return "virtual mode and hstatus.VTVM set -> virtual instruction, no TLB flush"
        return "virtual mode allowed -> no exception, TLB flushed"

    if row["mstatus_tvm"] == 1:
        if row["prv"] == 3:
            return "non virtual mode, mstatus.TVM set, M privilege -> no exception, TLB flushed"
        return "non virtual mode, mstatus.TVM set, privilege below M -> illegal instruction, no TLB flush"

    if row["prv"] >= 1:
        return "non virtual mode, mstatus.TVM clear, S or M privilege -> no exception, TLB flushed"

    return "non virtual mode, mstatus.TVM clear, U privilege -> illegal instruction, no TLB flush"

rows = []

for path in sorted(glob.glob(f"{OUT_DIR}/test*.ktest")):
    d = parse_ktest(path)

    row = {
        "ktest": os.path.basename(path),
        "ext_s": read_i32(d.get("ext_s")),
        "has_mmu": read_i32(d.get("has_mmu")),
        "v": read_i32(d.get("v")),
        "prv": read_i32(d.get("prv")),
        "hstatus_vtvm": read_i32(d.get("hstatus_vtvm")),
        "mstatus_tvm": read_i32(d.get("mstatus_tvm")),
        "obs_exc": read_i32(d.get("obs_exc")),
        "obs_tlb_flushed": read_i32(d.get("obs_tlb_flushed")),
    }

    rows.append(row)

df = pd.DataFrame(rows)

df["exc_name"] = df["obs_exc"].apply(exc_name)
df["path_summary"] = df.apply(explain, axis=1)

df = df.sort_values([
    "ext_s",
    "has_mmu",
    "v",
    "mstatus_tvm",
    "prv",
    "hstatus_vtvm",
    "ktest",
])

df.to_csv("spike_sfence_vma_klee_paths.csv", index=False)

print("num ktests:", len(df))
print("unique path summaries:", df["path_summary"].nunique())
print(df.to_string(index=False))

num ktests: 9
unique path summaries: 9
           ktest  ext_s  has_mmu  v  prv  hstatus_vtvm  mstatus_tvm  obs_exc  obs_tlb_flushed            exc_name                                                                              path_summary
test000001.ktest      0        0  0    0             0            0        2                0 illegal instruction                                 S extension disabled -> illegal instruction, no TLB flush
test000002.ktest      1        0  0    0             0            0        2                0 illegal instruction                                         MMU disabled -> illegal instruction, no TLB flush
test000006.ktest      1        1  0    0             0            0        2                0 illegal instruction     non virtual mode, mstatus.TVM clear, U privilege -> illegal instruction, no TLB flush
test000009.ktest      1        1  0    1             0            0        0                1                none        non virtual mode, mstatu